# LeetCode 733 — Flood Fill bằng Python

**Link đề:** https://leetcode.com/problems/flood-fill/
**Độ khó:** Easy · **Chủ đề:** Array, Depth-First Search, Breadth-First Search, Matrix

```python
class Solution:
    def floodFill(self, image: list[list[int]], sr: int, sc: int, color: int) -> list[list[int]]:
        ...
```

## Mục tiêu học
1. Hiểu ma trận (grid) như một **đồ thị**: mỗi ô là một đỉnh, nối với 4 ô kề trên/dưới/trái/phải.
2. Ôn `list` 2 chiều, chỉ số `image[r][c]`, kiểm tra biên, list hướng `DIRS`.
3. Học 3 cách duyệt vùng liên thông: **DFS đệ quy** → **DFS bằng stack** → **BFS bằng `deque`**.
4. Nắm cái bẫy kinh điển: **màu mới trùng màu cũ** → lặp vô hạn.
5. Trace trạng thái lưới qua từng bước, chứng minh, phân tích `O(m·n)`, làm bài tập (Number of Islands, đếm vùng...).

## 1. Đề bài

Ảnh là lưới `m × n` số nguyên `image`, `image[r][c]` là màu của pixel hàng `r`, cột `c`. Cho điểm bắt đầu `(sr, sc)` và màu mới `color`.

**Flood fill** (công cụ "thùng sơn" trong Paint):
1. Gọi `orig = image[sr][sc]` là màu ban đầu của pixel xuất phát.
2. Tô pixel xuất phát thành `color`.
3. Lan sang các pixel **kề cạnh** (trên, dưới, trái, phải — **không** tính chéo) có màu `orig`, tô chúng, rồi tiếp tục lan từ chúng.
4. Dừng khi không còn pixel nào màu `orig` nối được tới vùng đã tô.

Trả về ảnh sau khi tô.

**Ví dụ 1:** `image = [[1,1,1],[1,1,0],[1,0,1]]`, `sr = 1, sc = 1`, `color = 2` → `[[2,2,2],[2,2,0],[2,0,1]]`.
Góc dưới phải vẫn là `1` vì chỉ nối **chéo** với vùng, không nối cạnh.

**Ví dụ 2:** `image = [[0,0,0],[0,0,0]]`, `sr = 0, sc = 0`, `color = 0` → không đổi (màu mới trùng màu cũ).

**Constraints:** `1 <= m, n <= 50`, `0 <= image[i][j], color < 2^16`, `0 <= sr < m`, `0 <= sc < n`.

## 2. Nền tảng Python

### 2.1. List 2 chiều (list của các list)

`image[r]` là **hàng** `r` (một list), `image[r][c]` là ô ở hàng `r`, cột `c`. Số hàng `m = len(image)`, số cột `n = len(image[0])`.

In [1]:
image = [[1, 1, 1],
         [1, 1, 0],
         [1, 0, 1]]
print("m =", len(image), "| n =", len(image[0]))
print("hàng 1:", image[1])
print("ô (1, 2):", image[1][2])

m = 3 | n = 3
hàng 1: [1, 1, 0]
ô (1, 2): 0


### 2.2. Hàm in lưới cho dễ nhìn

Ta viết sẵn `show(grid)` để in từng hàng thẳng cột — dùng suốt notebook khi trace.

In [2]:
def show(grid, title=None):
    if title:
        print(title)
    for row in grid:
        print("   " + " ".join(f"{v:>2}" for v in row))

show(image, "Ảnh ví dụ 1:")

Ảnh ví dụ 1:
    1  1  1
    1  1  0
    1  0  1


### 2.3. Bẫy khi sao chép list 2 chiều

LeetCode cho phép sửa trực tiếp `image`. Nhưng trong notebook ta muốn chạy nhiều cách trên **cùng** một ảnh gốc, nên phải **sao chép sâu**:
- `copy = image` → chỉ là tên khác của **cùng** object (sửa một bên, bên kia đổi theo).
- `[row[:] for row in image]` → sao chép từng hàng → độc lập.

Ngoài ra `[[0] * 3] * 2` tạo 2 hàng là **cùng một list** — bẫy kinh điển khi tạo lưới.

In [3]:
a = [[1, 1], [1, 1]]
alias = a
deep = [row[:] for row in a]
alias[0][0] = 9
print("a     =", a)       # bị đổi theo alias
print("deep  =", deep)    # không đổi

bad = [[0] * 3] * 2
bad[0][0] = 7
print("bad   =", bad)     # cả 2 hàng đều đổi!
good = [[0] * 3 for _ in range(2)]
good[0][0] = 7
print("good  =", good)

a     = [[9, 1], [1, 1]]
deep  = [[1, 1], [1, 1]]
bad   = [[7, 0, 0], [7, 0, 0]]
good  = [[7, 0, 0], [0, 0, 0]]


### 2.4. Bốn hướng và kiểm tra biên

Thay vì viết 4 khối `if` gần giống nhau, dùng list các **độ lệch** `(dr, dc)`:

```python
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải
```

Ô kề của `(r, c)` là `(r + dr, c + dc)`. Ô hợp lệ khi `0 <= nr < m and 0 <= nc < n` (Python cho phép viết so sánh nối chuỗi `0 <= nr < m`).

In [4]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
m, n = 3, 3
r, c = 0, 2      # góc trên phải
for dr, dc in DIRS:
    nr, nc = r + dr, c + dc
    inside = 0 <= nr < m and 0 <= nc < n
    print(f"({r},{c}) + ({dr:>2},{dc:>2}) -> ({nr:>2},{nc:>2})  {'hợp lệ' if inside else 'ra ngoài'}")

(0,2) + (-1, 0) -> (-1, 2)  ra ngoài
(0,2) + ( 1, 0) -> ( 1, 2)  hợp lệ
(0,2) + ( 0,-1) -> ( 0, 1)  hợp lệ
(0,2) + ( 0, 1) -> ( 0, 3)  ra ngoài


### 2.5. Lưới là một đồ thị

Mỗi ô là một **đỉnh**; hai ô **kề cạnh và cùng màu `orig`** thì có **cạnh** nối. Flood fill = tìm **thành phần liên thông** chứa `(sr, sc)` rồi tô hết. Mọi thuật toán duyệt đồ thị (DFS, BFS) đều dùng được.

Vùng cần tô ở ví dụ 1 (đánh dấu `*`):

```text
 *  *  *
 *  *  0
 *  0  1      <- ô (2,2) cùng màu 1 nhưng chỉ chạm chéo -> không thuộc vùng
```

## 3. Cách nghĩ trực giác và cái bẫy lặp vô hạn

Cách nghĩ tự nhiên: "tô ô hiện tại, rồi làm y như vậy với 4 ô xung quanh nếu chúng có màu `orig`". Đây chính là **đệ quy**.

Vì sao không bị đi vòng vòng mãi? Vì ô đã tô có màu `color` ≠ `orig`, nên lần sau gặp lại nó **không còn thỏa** điều kiện "màu `orig`" → tự động đóng vai trò đánh dấu "đã thăm".

**Nhưng** nếu `color == orig`, ô vừa tô vẫn "có màu `orig`" → hai ô kề nhau gọi qua lại mãi → đệ quy vô hạn (`RecursionError`). Vì vậy dòng đầu tiên của mọi lời giải:

```python
if orig == color:
    return image        # không có gì để đổi
```

## 4. Cách 1 — DFS đệ quy

DFS (Depth-First Search, duyệt theo chiều sâu): đi sâu theo một hướng tới khi bế tắc rồi mới quay lui.

Hàm `dfs(r, c)`:
1. Nếu `(r, c)` ra ngoài lưới → dừng.
2. Nếu `image[r][c] != orig` → dừng (khác màu hoặc đã tô).
3. Tô `image[r][c] = color`.
4. Gọi `dfs` cho 4 ô kề.

Hàm lồng (nested function) `dfs` đọc được biến `image, orig, color, m, n` của hàm ngoài — không cần truyền lại.

In [5]:
def flood_fill_dfs_recursive(image, sr, sc, color):
    m, n = len(image), len(image[0])
    orig = image[sr][sc]
    if orig == color:
        return image

    def dfs(r, c):
        if not (0 <= r < m and 0 <= c < n):
            return
        if image[r][c] != orig:
            return
        image[r][c] = color
        for dr, dc in DIRS:
            dfs(r + dr, c + dc)

    dfs(sr, sc)
    return image

img = [[1, 1, 1], [1, 1, 0], [1, 0, 1]]
show(flood_fill_dfs_recursive([row[:] for row in img], 1, 1, 2), "Kết quả cách 1:")
show(flood_fill_dfs_recursive([[0, 0, 0], [0, 0, 0]], 0, 0, 0), "Ví dụ 2:")

Kết quả cách 1:
    2  2  2
    2  2  0
    2  0  1
Ví dụ 2:
    0  0  0
    0  0  0


### Giới hạn độ sâu đệ quy

Python mặc định giới hạn độ sâu gọi hàm (`sys.getrecursionlimit()`, thường là 1000–3000). Trường hợp xấu nhất của bài: lưới `50 × 50` cùng màu → vùng có `2500` ô, và DFS có thể đi sâu gần `2500` tầng. Cell dưới **đo** độ sâu lớn nhất thật sự (đếm bằng biến, không gây lỗi).

In [6]:
import sys
print("recursion limit hiện tại:", sys.getrecursionlimit())

def max_dfs_depth(m, n):
    seen = [[False] * n for _ in range(m)]
    best = 0
    def dfs(r, c, depth):
        nonlocal best
        if not (0 <= r < m and 0 <= c < n) or seen[r][c]:
            return
        seen[r][c] = True
        best = max(best, depth)
        for dr, dc in DIRS:
            dfs(r + dr, c + dc, depth + 1)
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(10_000)
    try:
        dfs(0, 0, 1)
    finally:
        sys.setrecursionlimit(old)
    return best

print("Độ sâu đệ quy lớn nhất trên lưới 50x50 cùng màu:", max_dfs_depth(50, 50))

recursion limit hiện tại: 1000
Độ sâu đệ quy lớn nhất trên lưới 50x50 cùng màu: 2500


Độ sâu lớn có thể vượt giới hạn mặc định → `RecursionError` trên một số môi trường. Hai cách xử lý:
- Tăng giới hạn: `sys.setrecursionlimit(10_000)` (LeetCode thường đã đặt cao, nhưng đừng phụ thuộc).
- **Bỏ đệ quy**, tự quản lý một **stack** → Cách 2.

`nonlocal best` ở trên cho phép hàm lồng **gán lại** biến của hàm ngoài (chỉ đọc thì không cần, gán thì cần).

## 5. Cách 2 — DFS bằng stack tường minh (không đệ quy)

**Stack** (ngăn xếp) = "vào sau ra trước" (LIFO). Trong Python dùng `list`: `append` để đẩy vào, `pop()` để lấy phần tử **cuối**.

In [7]:
stack = []
stack.append("A"); stack.append("B"); stack.append("C")
print(stack)
print(stack.pop(), stack.pop())   # C B  (vào sau ra trước)
print(stack)

['A', 'B', 'C']
C B
['A']


Thuật toán:
1. Nếu `orig == color` → trả nguyên ảnh.
2. Tô ô xuất phát, đẩy vào stack.
3. Lặp khi stack còn: `pop` một ô; với mỗi ô kề **trong lưới** và **màu `orig`** → tô ngay rồi đẩy vào stack.

**Tô ngay khi đẩy** (không đợi lúc lấy ra) để một ô không bị đẩy vào nhiều lần.

In [8]:
def flood_fill_dfs_stack(image, sr, sc, color):
    m, n = len(image), len(image[0])
    orig = image[sr][sc]
    if orig == color:
        return image
    image[sr][sc] = color
    stack = [(sr, sc)]
    while stack:
        r, c = stack.pop()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == orig:
                image[nr][nc] = color
                stack.append((nr, nc))
    return image

show(flood_fill_dfs_stack([row[:] for row in img], 1, 1, 2), "Kết quả cách 2:")

Kết quả cách 2:
    2  2  2
    2  2  0
    2  0  1


## 6. Cách 3 — BFS bằng `collections.deque` (lời giải chính)

BFS (Breadth-First Search, duyệt theo chiều rộng): lan ra **theo từng lớp** — giống vết loang của giọt mực. Cấu trúc dữ liệu là **queue** (hàng đợi, "vào trước ra trước", FIFO).

`list.pop(0)` lấy phần tử đầu nhưng tốn `O(len)` vì phải dời mọi phần tử. `deque` (double-ended queue) có `popleft()` **`O(1)`**.

In [9]:
from collections import deque

q = deque()
q.append("A"); q.append("B"); q.append("C")
print(q)
print(q.popleft(), q.popleft())   # A B  (vào trước ra trước)
print(q, "| còn phần tử?", bool(q))

deque(['A', 'B', 'C'])
A B
deque(['C']) | còn phần tử? True


In [10]:
def flood_fill_bfs(image, sr, sc, color):
    m, n = len(image), len(image[0])
    orig = image[sr][sc]
    if orig == color:
        return image
    image[sr][sc] = color
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == orig:
                image[nr][nc] = color
                q.append((nr, nc))
    return image

show(flood_fill_bfs([row[:] for row in img], 1, 1, 2), "Kết quả cách 3:")

Kết quả cách 3:
    2  2  2
    2  2  0
    2  0  1


Nhận xét: Cách 2 và Cách 3 chỉ khác **một dòng** (`stack.pop()` ↔ `q.popleft()`). Kết quả tô giống hệt nhau — chỉ **thứ tự** tô khác. Với flood fill, thứ tự không quan trọng; nhưng BFS có thêm ưu điểm: ô được lấy ra theo **khoảng cách tăng dần** tới điểm xuất phát (dùng cho bài đường đi ngắn nhất).

## 7. Trace từng bước

Ta trace BFS trên ví dụ 1, in hàng đợi và lưới sau mỗi lần lấy một ô ra.

| Bước | Lấy ra | Ô kề được tô (màu 1 → 2) | Queue sau bước |
|---:|---|---|---|
| 0 | — | (1,1) tô lúc khởi tạo | [(1,1)] |
| 1 | (1,1) | (0,1), (1,0) — (2,1)=0, (1,2)=0 bỏ | [(0,1),(1,0)] |
| 2 | (0,1) | (0,0), (0,2) | [(1,0),(0,0),(0,2)] |
| 3 | (1,0) | (2,0) | [(0,0),(0,2),(2,0)] |
| 4–6 | (0,0), (0,2), (2,0) | không còn ô màu 1 kề | [] |

Ô `(2,2)` không bao giờ được xét vì các ô kề cạnh của nó (`(1,2)` và `(2,1)`) đều màu 0.

In [11]:
def trace_flood_bfs(image, sr, sc, color):
    image = [row[:] for row in image]
    m, n = len(image), len(image[0])
    orig = image[sr][sc]
    print(f"orig = {orig}, color = {color}")
    if orig == color:
        print("Màu mới trùng màu cũ -> trả nguyên ảnh")
        return image
    image[sr][sc] = color
    q = deque([(sr, sc)])
    step = 0
    show(image, f"Bước {step}: tô ({sr},{sc}), queue = {list(q)}")
    while q:
        r, c = q.popleft()
        step += 1
        painted = []
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == orig:
                image[nr][nc] = color
                q.append((nr, nc))
                painted.append((nr, nc))
        show(image, f"Bước {step}: lấy ({r},{c}), tô thêm {painted}, queue = {list(q)}")
    return image

trace_flood_bfs([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2)

orig = 1, color = 2
Bước 0: tô (1,1), queue = [(1, 1)]
    1  1  1
    1  2  0
    1  0  1
Bước 1: lấy (1,1), tô thêm [(0, 1), (1, 0)], queue = [(0, 1), (1, 0)]
    1  2  1
    2  2  0
    1  0  1
Bước 2: lấy (0,1), tô thêm [(0, 0), (0, 2)], queue = [(1, 0), (0, 0), (0, 2)]
    2  2  2
    2  2  0
    1  0  1
Bước 3: lấy (1,0), tô thêm [(2, 0)], queue = [(0, 0), (0, 2), (2, 0)]
    2  2  2
    2  2  0
    2  0  1
Bước 4: lấy (0,0), tô thêm [], queue = [(0, 2), (2, 0)]
    2  2  2
    2  2  0
    2  0  1
Bước 5: lấy (0,2), tô thêm [], queue = [(2, 0)]
    2  2  2
    2  2  0
    2  0  1
Bước 6: lấy (2,0), tô thêm [], queue = []
    2  2  2
    2  2  0
    2  0  1


[[2, 2, 2], [2, 2, 0], [2, 0, 1]]

### So sánh thứ tự tô của DFS-stack và BFS

Ghi lại thứ tự các ô được tô trên lưới `4 × 4` toàn số 0, bắt đầu từ góc `(0, 0)`. BFS tô theo "đường chéo" (khoảng cách tăng dần), DFS-stack chạy dài theo một hướng trước.

In [12]:
def paint_order(grid_m, grid_n, use_bfs):
    image = [[0] * grid_n for _ in range(grid_m)]
    order = [[0] * grid_n for _ in range(grid_m)]
    k = 1
    order[0][0] = k
    image[0][0] = 1
    box = deque([(0, 0)])
    while box:
        r, c = box.popleft() if use_bfs else box.pop()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < grid_m and 0 <= nc < grid_n and image[nr][nc] == 0:
                image[nr][nc] = 1
                k += 1
                order[nr][nc] = k
                box.append((nr, nc))
    return order

show(paint_order(4, 4, use_bfs=True), "Thứ tự tô BFS:")
show(paint_order(4, 4, use_bfs=False), "Thứ tự tô DFS (stack):")

Thứ tự tô BFS:
    1  3  6 10
    2  5  9 13
    4  8 12 15
    7 11 14 16
Thứ tự tô DFS (stack):
    1  3  5  7
    2  4  6  8
   15 13 11  9
   16 14 12 10


## 8. Code nộp LeetCode

### BFS (khuyến nghị — không lo giới hạn đệ quy)

In [13]:
from collections import deque


class Solution:
    def floodFill(self, image: list[list[int]], sr: int, sc: int, color: int) -> list[list[int]]:
        orig = image[sr][sc]
        if orig == color:
            return image
        m, n = len(image), len(image[0])
        image[sr][sc] = color
        q = deque([(sr, sc)])
        while q:
            r, c = q.popleft()
            for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
                if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == orig:
                    image[nr][nc] = color
                    q.append((nr, nc))
        return image


print(Solution().floodFill([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2))
print(Solution().floodFill([[0, 0, 0], [0, 0, 0]], 0, 0, 0))

[[2, 2, 2], [2, 2, 0], [2, 0, 1]]
[[0, 0, 0], [0, 0, 0]]


### DFS đệ quy (ngắn, đúng tinh thần hint)

Khi nộp trên LeetCode đổi tên class thành `Solution`.

In [14]:
class SolutionRecursive:
    def floodFill(self, image: list[list[int]], sr: int, sc: int, color: int) -> list[list[int]]:
        orig = image[sr][sc]
        if orig == color:
            return image
        m, n = len(image), len(image[0])

        def dfs(r: int, c: int) -> None:
            if 0 <= r < m and 0 <= c < n and image[r][c] == orig:
                image[r][c] = color
                dfs(r - 1, c); dfs(r + 1, c); dfs(r, c - 1); dfs(r, c + 1)

        dfs(sr, sc)
        return image


print(SolutionRecursive().floodFill([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2))

[[2, 2, 2], [2, 2, 0], [2, 0, 1]]


## 9. Vì sao đúng?

Gọi `R` là **vùng đúng**: tập ô có màu `orig` nối được với `(sr, sc)` qua các bước kề cạnh đi trên ô màu `orig` (giả sử `orig != color`).

**Invariant (BFS/DFS-stack):** mọi ô đã bị tô đều thuộc `R`, và mọi ô trong queue/stack đều đã bị tô.

1. **Không tô thừa:** một ô chỉ bị tô khi nó có màu `orig` và kề một ô đã tô (đã thuộc `R`) → nó cũng thuộc `R`.
2. **Không bỏ sót:** giả sử ô `x ∈ R` không bị tô. Xét đường đi từ `(sr, sc)` tới `x` trong `R`; lấy ô **đầu tiên** trên đường chưa bị tô, gọi là `y`, và ô liền trước `p` đã tô. `p` đã từng vào queue và được lấy ra (vòng lặp chạy tới khi rỗng); lúc đó `y` vẫn màu `orig` (chưa tô) nên `y` phải được tô — mâu thuẫn.
3. **Dừng:** mỗi ô được đẩy vào **tối đa 1 lần** (vì tô ngay khi đẩy và ô đã tô không còn màu `orig`), nên vòng lặp kết thúc sau tối đa `m·n` lần lấy.

Trường hợp `orig == color`: vùng tô xong vẫn y nguyên, nên trả ảnh gốc là đúng — và bắt buộc phải trả sớm để đảm bảo điều 3.

## 10. Độ phức tạp

Gọi `N = m · n` (số pixel), `K = |R|` (số pixel thực sự bị tô, `K ≤ N`).

| Cách | Time | Space phụ |
|---|---|---|
| 1. DFS đệ quy | `O(K)` ⊆ `O(N)` | `O(K)` call stack (tới ~`N` tầng) |
| 2. DFS stack | `O(K)` | `O(K)` stack |
| 3. BFS deque | `O(K)` | `O(K)` queue (thực tế thường nhỏ hơn: cỡ "đường biên" của vùng) |

Mỗi ô trong `R` được lấy ra một lần và xét 4 hàng xóm → `4K` phép kiểm. Ta sửa trực tiếp `image` nên không tốn thêm `O(N)` cho mảng `visited`.

Với `N ≤ 2500`, mọi cách đều rất nhanh; điểm khác biệt thực tế duy nhất là rủi ro giới hạn đệ quy của Cách 1.

## 11. Bộ test

In [15]:
import copy

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}")

def run(f, image, sr, sc, color):
    return f(copy.deepcopy(image), sr, sc, color)

sol = Solution()
check("đề 1", run(sol.floodFill, [[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2), [[2, 2, 2], [2, 2, 0], [2, 0, 1]])
check("đề 2 (màu trùng)", run(sol.floodFill, [[0, 0, 0], [0, 0, 0]], 0, 0, 0), [[0, 0, 0], [0, 0, 0]])
check("1x1", run(sol.floodFill, [[5]], 0, 0, 7), [[7]])
check("1x1 màu trùng", run(sol.floodFill, [[5]], 0, 0, 5), [[5]])
check("ô cô lập", run(sol.floodFill, [[0, 1, 0], [1, 0, 1], [0, 1, 0]], 1, 1, 3), [[0, 1, 0], [1, 3, 1], [0, 1, 0]])
check("chéo không lan", run(sol.floodFill, [[1, 0], [0, 1]], 0, 0, 9), [[9, 0], [0, 1]])
check("1 hàng", run(sol.floodFill, [[2, 2, 3, 2]], 0, 0, 4), [[4, 4, 3, 2]])
check("1 cột", run(sol.floodFill, [[2], [2], [3], [2]], 3, 0, 1), [[2], [2], [3], [1]])
check("xoắn ốc", run(sol.floodFill,
      [[1, 1, 1, 1], [0, 0, 0, 1], [1, 1, 0, 1], [1, 1, 1, 1]], 0, 0, 8),
      [[8, 8, 8, 8], [0, 0, 0, 8], [8, 8, 0, 8], [8, 8, 8, 8]])

PASS  đề 1
PASS  đề 2 (màu trùng)
PASS  1x1
PASS  1x1 màu trùng
PASS  ô cô lập
PASS  chéo không lan
PASS  1 hàng
PASS  1 cột
PASS  xoắn ốc


### Đối chiếu 5 cài đặt trên lưới ngẫu nhiên

Lưới nhỏ, màu chỉ `0..2` để tạo nhiều vùng liên thông thú vị. Seed cố định để lặp lại được.

In [16]:
import random

rng = random.Random(733)
impls = [flood_fill_dfs_recursive, flood_fill_dfs_stack, flood_fill_bfs,
         Solution().floodFill, SolutionRecursive().floodFill]
changed = 0
for _ in range(1500):
    m_, n_ = rng.randint(1, 6), rng.randint(1, 6)
    grid = [[rng.randint(0, 2) for _ in range(n_)] for _ in range(m_)]
    sr_, sc_, col = rng.randrange(m_), rng.randrange(n_), rng.randint(0, 2)
    outs = [run(f, grid, sr_, sc_, col) for f in impls]
    assert all(o == outs[0] for o in outs)
    changed += outs[0] != grid
print("1500 lưới ngẫu nhiên: 5 cài đặt khớp nhau; số lần ảnh bị thay đổi:", changed)

1500 lưới ngẫu nhiên: 5 cài đặt khớp nhau; số lần ảnh bị thay đổi: 1002


In [17]:
big = [[7] * 50 for _ in range(50)]
res = run(Solution().floodFill, big, 25, 25, 1)
print("Lưới 50x50 toàn 7 -> toàn 1?", all(v == 1 for row in res for v in row))

Lưới 50x50 toàn 7 -> toàn 1? True


## 12. Lỗi người mới hay gặp

### Lỗi 1: Không xử lý `orig == color` → lặp vô hạn

Với đệ quy là `RecursionError`; với stack/queue là vòng `while` không bao giờ dừng (nguy hiểm hơn vì treo máy). Cell dưới cố tình **giới hạn số bước** để minh họa mà không treo.

In [18]:
def wrong_no_same_color_check(image, sr, sc, color, max_steps=50):
    m, n = len(image), len(image[0])
    orig = image[sr][sc]
    image[sr][sc] = color
    stack, steps = [(sr, sc)], 0
    while stack:
        steps += 1
        if steps > max_steps:
            return f"Đã chạy {max_steps} bước mà chưa dừng -> lặp vô hạn!"
        r, c = stack.pop()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == orig:
                image[nr][nc] = color
                stack.append((nr, nc))
    return image

print(wrong_no_same_color_check([[0, 0], [0, 0]], 0, 0, 0))

Đã chạy 50 bước mà chưa dừng -> lặp vô hạn!


**Sửa:** `if orig == color: return image` ngay đầu hàm.

### Lỗi 2: So sánh với `color` thay vì `orig`, hoặc đọc `orig` **sau** khi đã tô

Nếu viết `image[sr][sc] = color` trước rồi mới `orig = image[sr][sc]`, thì `orig` đã thành màu mới → không lan đi đâu cả.

In [19]:
def wrong_read_orig_late(image, sr, sc, color):
    m, n = len(image), len(image[0])
    image[sr][sc] = color
    orig = image[sr][sc]            # SAI: đã bị ghi đè thành color
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == orig:
                image[nr][nc] = color
                q.append((nr, nc))
    return image

print("sai :", wrong_read_orig_late([[1, 1], [1, 0]], 0, 0, 2))
print("đúng:", flood_fill_bfs([[1, 1], [1, 0]], 0, 0, 2))

sai : [[2, 1], [1, 0]]
đúng: [[2, 2], [2, 0]]


### Lỗi 3: Lan cả 8 hướng (tính cả chéo)

Đề chỉ cho 4 hướng kề **cạnh**. Thêm hướng chéo sẽ tô nhầm góc `(2,2)` của ví dụ 1.

In [20]:
DIRS8 = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]

def wrong_eight_dirs(image, sr, sc, color):
    m, n = len(image), len(image[0])
    orig = image[sr][sc]
    if orig == color:
        return image
    image[sr][sc] = color
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS8:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == orig:
                image[nr][nc] = color
                q.append((nr, nc))
    return image

show(wrong_eight_dirs([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2), "sai (8 hướng):")

sai (8 hướng):
    2  2  2
    2  2  0
    2  0  2


### Lỗi 4: Kiểm tra biên sau khi truy cập, hoặc dùng chỉ số âm

`image[-1][c]` **không lỗi** trong Python mà trả về **hàng cuối** → lan "xuyên tường" sang mép bên kia. Phải kiểm `0 <= nr` **trước** khi đọc `image[nr][nc]` (ngắn mạch `and` lo việc này).

In [21]:
row = [10, 20, 30]
print("row[-1] =", row[-1], "<- không báo lỗi, nhưng là phần tử cuối!")

row[-1] =

 30 <- không báo lỗi, nhưng là phần tử cuối!


### Lỗi 5: Đánh dấu khi lấy ra thay vì khi đẩy vào (BFS)

Nếu chỉ tô lúc `popleft`, một ô có thể bị đẩy vào queue nhiều lần (từ nhiều hàng xóm) → vẫn đúng kết quả nhưng tốn gấp nhiều lần. Quy tắc: **tô/đánh dấu ngay khi đẩy vào**.

## 13. Mẹo nhận dạng pattern

- Thấy **lưới 2D** + "vùng", "liên thông", "lan", "đảo (island)", "tô màu", "bị bao quanh" → **DFS/BFS trên grid**.
- Khung code luôn có: `m, n`, `DIRS`, kiểm biên `0 <= nr < m and 0 <= nc < n`, điều kiện "được đi" (cùng màu, là đất...), **đánh dấu đã thăm** (sửa trực tiếp hoặc mảng `visited`).
- Cần **đường đi ngắn nhất** (số bước tối thiểu) → BFS. Chỉ cần "đi được tới đâu" → DFS hay BFS đều được.
- Bài cùng họ: 200 Number of Islands, 695 Max Area of Island, 463 Island Perimeter, 130 Surrounded Regions, 994 Rotting Oranges (BFS nhiều nguồn), 1034 Coloring A Border.

## 14. Biến thể

### 14.1. Đếm số pixel bị tô (diện tích vùng)

Chỉ cần đếm số lần tô — nền tảng cho 695 Max Area of Island.

In [22]:
def region_size(image, sr, sc):
    image = [row[:] for row in image]
    m, n = len(image), len(image[0])
    orig = image[sr][sc]
    MARK = None                    # dùng một giá trị chắc chắn khác mọi màu
    image[sr][sc] = MARK
    q, size = deque([(sr, sc)]), 1
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == orig:
                image[nr][nc] = MARK
                size += 1
                q.append((nr, nc))
    return size

print(region_size([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1))   # 6
print(region_size([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 2, 2))   # 1

6
1


Mẹo: dùng `None` làm dấu → không bao giờ trùng `orig` (là số), nên không cần xử lý riêng trường hợp "màu trùng".

### 14.2. Đếm số vùng liên thông (khung của Number of Islands)

Duyệt mọi ô; gặp ô chưa thăm → tăng bộ đếm và flood fill cả vùng của nó để không đếm lại.

In [23]:
def count_regions(image):
    m, n = len(image), len(image[0])
    seen = [[False] * n for _ in range(m)]
    regions = 0
    for i in range(m):
        for j in range(n):
            if seen[i][j]:
                continue
            regions += 1
            color0 = image[i][j]
            seen[i][j] = True
            q = deque([(i, j)])
            while q:
                r, c = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc] and image[nr][nc] == color0:
                        seen[nr][nc] = True
                        q.append((nr, nc))
    return regions

print(count_regions([[1, 1, 1], [1, 1, 0], [1, 0, 1]]))   # vùng 1 lớn, hai ô 0 riêng, ô 1 góc -> 4

4


Ở đây ta dùng mảng `seen` riêng thay vì sửa `image`, vì cần giữ màu gốc để so sánh từng vùng.

### 14.3. BFS theo lớp: khoảng cách từ điểm xuất phát

Thêm `dist` vào mỗi phần tử queue — biến flood fill thành "bản đồ khoảng cách". Đây là cầu nối tới bài đường đi ngắn nhất trên lưới.

In [24]:
def distance_map(image, sr, sc):
    m, n = len(image), len(image[0])
    orig = image[sr][sc]
    dist = [[-1] * n for _ in range(m)]
    dist[sr][sc] = 0
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and dist[nr][nc] == -1 and image[nr][nc] == orig:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
    return dist

show(distance_map([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1), "Khoảng cách (−1 = không tới được):")

Khoảng cách (−1 = không tới được):
    2  1  2
    1  0 -1
    2 -1 -1


## 15. Bài tập

### Bài 1 — Tính tay
Với `image = [[0,0,1],[1,0,1],[1,0,0]]`, `sr = 0, sc = 0`, `color = 5`, ảnh kết quả là gì? Tự vẽ rồi đối chiếu đáp án.

### Bài 2 — Liệt kê ô kề hợp lệ
Viết `neighbors(r, c, m, n)` trả list các ô kề cạnh nằm trong lưới, theo thứ tự lên, xuống, trái, phải.

In [25]:
def neighbors(r: int, c: int, m: int, n: int) -> list[tuple[int, int]]:
    # TODO: dùng DIRS và kiểm tra biên
    pass

# assert neighbors(0, 0, 3, 3) == [(1, 0), (0, 1)]
# assert neighbors(1, 1, 3, 3) == [(0, 1), (2, 1), (1, 0), (1, 2)]
# assert neighbors(0, 0, 1, 1) == []

### Bài 3 — Điền chỗ trống BFS flood fill

In [26]:
def exercise_flood_fill(image, sr, sc, color):
    m, n = len(image), len(image[0])
    orig = image[sr][sc]
    # TODO 1: nếu orig == color thì trả image
    image[sr][sc] = color
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            # TODO 2: nếu (nr, nc) trong lưới và có màu orig -> tô + đẩy vào q
            pass
    return image

# assert exercise_flood_fill([[1,1,1],[1,1,0],[1,0,1]], 1, 1, 2) == [[2,2,2],[2,2,0],[2,0,1]]
# assert exercise_flood_fill([[0,0,0],[0,0,0]], 0, 0, 0) == [[0,0,0],[0,0,0]]

### Bài 4 — Max Area of Island (LeetCode 695, rút gọn)
`grid` gồm 0 (nước) và 1 (đất). Trả diện tích (số ô) của hòn đảo lớn nhất (đảo = vùng ô 1 liên thông 4 hướng). Không có đảo → `0`.

In [27]:
def max_area_of_island(grid: list[list[int]]) -> int:
    # TODO
    pass

# assert max_area_of_island([[0,1,1],[1,1,0],[0,0,1]]) == 4
# assert max_area_of_island([[0,0],[0,0]]) == 0

### Bài 5 — Tô đường biên của vùng (ý tưởng LeetCode 1034)
Tô màu `color` cho những ô thuộc vùng của `(sr, sc)` mà **nằm ở biên** của vùng: ô ở mép lưới, hoặc có ít nhất một ô kề cạnh **không** thuộc vùng. Ô bên trong vùng giữ nguyên.

In [28]:
def color_border(grid: list[list[int]], sr: int, sc: int, color: int) -> list[list[int]]:
    # TODO
    pass

# assert color_border([[1,1,1],[1,1,1],[1,1,1]], 1, 1, 2) == [[2,2,2],[2,1,2],[2,2,2]]
# assert color_border([[1,1],[1,2]], 0, 0, 3) == [[3,3],[3,2]]

### Bài 6 — Mini challenge: Surrounded Regions (LeetCode 130, rút gọn)
Lưới gồm `'X'` và `'O'`. Mọi vùng `'O'` **không** chạm mép lưới bị bao quanh → đổi thành `'X'`. Vùng `'O'` chạm mép giữ nguyên. Trả lưới mới.

In [29]:
def solve_surrounded(board: list[list[str]]) -> list[list[str]]:
    # TODO
    pass

# board = [list("XXXX"), list("XOOX"), list("XXOX"), list("XOXX")]
# assert solve_surrounded(board) == [list("XXXX"), list("XXXX"), list("XXXX"), list("XOXX")]

## 16. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Chỉ lan qua ô màu 0 kề cạnh | Đi từ (0,0) | (0,0)→(0,1)→(1,1)→(2,1)→(2,2) |
| 2 | 4 hướng, lọc ra ngoài | Vòng `for dr, dc in DIRS` | `if 0 <= nr < m and 0 <= nc < n: res.append((nr, nc))` |
| 3 | Như Cách 3 | Điều kiện kép | `if 0<=nr<m and 0<=nc<n and image[nr][nc]==orig:` tô + `q.append` |
| 4 | Mỗi đảo là một vùng | Duyệt mọi ô, gặp 1 thì BFS và đếm | Đổi ô đã thăm thành 0, giữ `best = max(best, size)` |
| 5 | Tìm vùng trước, sửa sau | BFS gom tập `region`; xét từng ô | ô biên nếu ở mép hoặc có hàng xóm `not in region` |
| 6 | Đảo ngược câu hỏi: ô nào **được cứu**? | Flood từ mọi `'O'` trên mép | Đánh dấu `'S'`; cuối cùng `'O'`→`'X'`, `'S'`→`'O'` |

## 17. Đáp án tham khảo

**Bài 1:** vùng màu 0 chứa `(0,0)` là `(0,0),(0,1),(1,1),(2,1),(2,2)` → `[[5,5,1],[1,5,1],[1,5,5]]`.

In [30]:
def answer_neighbors(r, c, m, n):
    res = []
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            res.append((nr, nc))
    return res


def answer_flood_fill(image, sr, sc, color):
    m, n = len(image), len(image[0])
    orig = image[sr][sc]
    if orig == color:
        return image
    image[sr][sc] = color
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for nr, nc in answer_neighbors(r, c, m, n):
            if image[nr][nc] == orig:
                image[nr][nc] = color
                q.append((nr, nc))
    return image


def answer_max_area(grid):
    grid = [row[:] for row in grid]
    m, n = len(grid), len(grid[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] != 1:
                continue
            grid[i][j] = 0
            q, size = deque([(i, j)]), 1
            while q:
                r, c = q.popleft()
                for nr, nc in answer_neighbors(r, c, m, n):
                    if grid[nr][nc] == 1:
                        grid[nr][nc] = 0
                        size += 1
                        q.append((nr, nc))
            best = max(best, size)
    return best


def answer_color_border(grid, sr, sc, color):
    m, n = len(grid), len(grid[0])
    orig = grid[sr][sc]
    region = {(sr, sc)}
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for nr, nc in answer_neighbors(r, c, m, n):
            if (nr, nc) not in region and grid[nr][nc] == orig:
                region.add((nr, nc))
                q.append((nr, nc))
    result = [row[:] for row in grid]
    for r, c in region:
        nbs = answer_neighbors(r, c, m, n)
        on_edge = len(nbs) < 4
        if on_edge or any(nb not in region for nb in nbs):
            result[r][c] = color
    return result


def answer_surrounded(board):
    board = [row[:] for row in board]
    m, n = len(board), len(board[0])
    q = deque()
    for i in range(m):
        for j in range(n):
            if (i in (0, m - 1) or j in (0, n - 1)) and board[i][j] == 'O':
                board[i][j] = 'S'
                q.append((i, j))
    while q:
        r, c = q.popleft()
        for nr, nc in answer_neighbors(r, c, m, n):
            if board[nr][nc] == 'O':
                board[nr][nc] = 'S'
                q.append((nr, nc))
    for i in range(m):
        for j in range(n):
            board[i][j] = 'O' if board[i][j] == 'S' else 'X'
    return board


print("Bài 1:", answer_flood_fill([[0, 0, 1], [1, 0, 1], [1, 0, 0]], 0, 0, 5))
assert answer_neighbors(0, 0, 3, 3) == [(1, 0), (0, 1)]
assert answer_neighbors(1, 1, 3, 3) == [(0, 1), (2, 1), (1, 0), (1, 2)]
assert answer_neighbors(0, 0, 1, 1) == []
assert answer_flood_fill([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2) == [[2, 2, 2], [2, 2, 0], [2, 0, 1]]
assert answer_flood_fill([[0, 0, 0], [0, 0, 0]], 0, 0, 0) == [[0, 0, 0], [0, 0, 0]]
assert answer_max_area([[0, 1, 1], [1, 1, 0], [0, 0, 1]]) == 4
assert answer_max_area([[0, 0], [0, 0]]) == 0
assert answer_color_border([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 1, 1, 2) == [[2, 2, 2], [2, 1, 2], [2, 2, 2]]
assert answer_color_border([[1, 1], [1, 2]], 0, 0, 3) == [[3, 3], [3, 2]]
board = [list("XXXX"), list("XOOX"), list("XXOX"), list("XOXX")]
assert answer_surrounded(board) == [list("XXXX"), list("XXXX"), list("XXXX"), list("XOXX")]
print("Tất cả đáp án PASS")

Bài 1: [[5, 5, 1], [1, 5, 1], [1, 5, 5]]
Tất cả đáp án PASS


Ghi chú bài 5: "ô ở mép lưới" ⟺ có ít hơn 4 hàng xóm hợp lệ (`len(nbs) < 4`). Ta tô vào **bản sao** `result` để việc tô không làm sai lệch phép kiểm "hàng xóm có thuộc vùng không" của các ô xét sau.

Ghi chú bài 6: kỹ thuật "**flood từ biên vào**" (multi-source BFS) — thay vì hỏi vùng nào bị bao, ta hỏi vùng nào thoát được ra biên. Đảo ngược câu hỏi là mẹo rất hay gặp.

## 18. Template nhớ nhanh + Checklist submit

```python
from collections import deque

class Solution:
    def floodFill(self, image, sr, sc, color):
        orig = image[sr][sc]
        if orig == color:                    # chặn lặp vô hạn
            return image
        m, n = len(image), len(image[0])
        image[sr][sc] = color                # tô ngay khi đẩy vào
        q = deque([(sr, sc)])
        while q:
            r, c = q.popleft()
            for nr, nc in ((r-1, c), (r+1, c), (r, c-1), (r, c+1)):
                if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == orig:
                    image[nr][nc] = color
                    q.append((nr, nc))
        return image
```

**Câu thần chú:** *"lưu orig — trùng thì về — tô khi đẩy — 4 hướng — kiểm biên trước khi đọc."*

**Checklist trước khi submit:**
- [ ] Đọc `orig = image[sr][sc]` **trước** khi tô?
- [ ] Có `if orig == color: return image`?
- [ ] Chỉ 4 hướng kề cạnh, không chéo?
- [ ] Kiểm `0 <= nr < m and 0 <= nc < n` trước khi truy cập (tránh chỉ số âm "vòng" ra cuối)?
- [ ] Tô/đánh dấu ngay khi đẩy vào queue/stack?
- [ ] Nếu dùng đệ quy: đã cân nhắc độ sâu tới ~2500 (`sys.setrecursionlimit`) hoặc chuyển sang BFS?